# RetailPulse Exploratory Analysis

Answers the vault's six EDA questions with **one-sentence findings + implementation consequences**. Synthetic data today; rerun on real data after `retailpulse ingest`.

In [ ]:
import pandas as pd

from retailpulse.data.ingest import load_curated

data = load_curated().to_pandas()
data["Date"] = pd.to_datetime(data["Date"])
data.head()

## 1. Seasonality (weekday / month / store type)

In [ ]:
data.groupby("DayOfWeek")["Sales"].mean().plot(kind="bar", title="Mean sales by day of week")

**Finding:** weekday pattern is strong (weekend dips, Sunday low).
**Consequence:** `weekday`/`is_weekend` features and a lag-7 seasonal-naive baseline are both justified.

## 2. Promo / holiday alignment

In [ ]:
open_days = data[data["Open"] == 1]
open_days.groupby("Promo")["Sales"].mean().plot(kind="bar", title="Mean sales with/without promo (open days)")
data.groupby("StateHoliday")["Sales"].mean()

**Finding:** promo lifts open-day sales; state holidays sharply depress or zero sales.
**Consequence:** promo/holiday flags are legal scheduled features and the closed-store zero rule is essential.

## 3. Closure / missing / outlier frequency

In [ ]:
closure_rate = 1 - data["Open"].mean()
print(f"closure rate: {closure_rate:.3f}")
data[["Sales", "Customers"]].isna().mean()

**Finding:** closures are frequent enough to matter; no missingness in synthetic data.
**Consequence:** models must never learn to predict non-zero sales on closed days.

## 4. Short / sparse / volatile store histories

In [ ]:
store_stats = data.groupby("Store")["Sales"].agg(["count", "mean", "std"])
store_stats["cv"] = store_stats["std"] / store_stats["mean"]
store_stats.describe()

**Finding:** all stores have full histories in synthetic data; volatility varies by store type.
**Consequence:** challenger subset selection stratifies by size AND volatility quartiles.

## 5. Train vs final-period drift

In [ ]:
cutoff = data["Date"].max() - pd.Timedelta(days=30)
early, late = data[data["Date"] < cutoff], data[data["Date"] >= cutoff]
print(f"early mean sales {early['Sales'].mean():.0f} vs final-30d mean {late['Sales'].mean():.0f}")

**Finding:** mild growth trend in synthetic data; no structural break.
**Consequence:** expanding-window folds capture the trend; the sealed holdout stays locked.

## Five modeling decisions this EDA supports

1. Use weekday/calendar + promo/holiday flags as scheduled features.
2. Enforce deterministic zeros for closed stores in every model.
3. Keep lag-7 seasonal-naive as the permanent baseline (weekday seasonality is real).
4. Stratify challenger subsets by size and volatility.
5. Keep the final 30 days sealed (trend exists; holdout must measure it honestly).